In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from matplotlib import rcParams
from tqdm import tqdm
import shap

In [ ]:
# ===================== DEVICE SETUP ===================== #
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Using device: {device}")

In [ ]:
# ===================== PATHS AND CONFIGURATION ===================== #
model_name = "convnext"
OUTPUT_PATH = f"1_Feature_Extraction/{model_name}"
RESULTS_PATH = os.path.join("4_Feature_Count_Comparison")
os.makedirs(RESULTS_PATH, exist_ok=True)

# Load feature data
try:
    train_df = pd.read_csv(os.path.join(OUTPUT_PATH, f"train_features_{model_name}.csv"))
    test_df = pd.read_csv(os.path.join(OUTPUT_PATH, f"test_features_{model_name}.csv"))
    print(f"Loaded {len(train_df)} training and {len(test_df)} testing samples")
except FileNotFoundError:
    print(f"ConvNext features not found, attempting to use available features")
    train_df = pd.read_csv(os.path.join("1_Feature_Extraction/densenet121", f"train_features_densenet121.csv"))
    test_df = pd.read_csv(os.path.join("1_Feature_Extraction/densenet121", f"test_features_densenet121.csv"))
    print(f"Using DenseNet features with {len(train_df)} training and {len(test_df)} testing samples")

In [ ]:
# Extract features and labels
feature_columns = [col for col in train_df.columns if col.startswith("feat_")]
X_train = train_df[feature_columns].values
y_train = train_df["label"].map({"glioma": 0, "meningioma": 1, "notumor": 2, "pituitary": 3}).values
X_test = test_df[feature_columns].values
y_test = test_df["label"].map({"glioma": 0, "meningioma": 1, "notumor": 2, "pituitary": 3}).values

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
N_CLASSES = len(CLASSES)  # Should be 4
print(f"Number of classes: {N_CLASSES}")

In [ ]:
# Plot settings
rcParams["font.family"] = "Times New Roman"
rcParams["axes.titlesize"] = 28
rcParams["axes.titlepad"] = 20
rcParams["axes.labelsize"] = 23
rcParams["xtick.labelsize"] = 18
rcParams["ytick.labelsize"] = 18
rcParams["legend.fontsize"] = 16
rcParams["lines.linewidth"] = 3
rcParams["axes.linewidth"] = 2

In [ ]:
# ===================== FEATURE SELECTION USING SHAP ===================== #
def apply_shap(X_train, X_test, n_features):
    print(f"Applying SHAP to select {n_features} features...")
    rf_model = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
    rf_model.fit(X_train, y_train)
    feature_importance = rf_model.feature_importances_
    selected_indices = np.argsort(feature_importance)[::-1][:n_features]
    selected_indices = np.array(selected_indices, dtype=int)
    X_train_selected = X_train[:, selected_indices]
    X_test_selected = X_test[:, selected_indices]
    return X_train_selected, X_test_selected, selected_indices

In [ ]:
# ===================== MODEL DEFINITION ===================== #
class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [ ]:
class Attention(nn.Module):
    def __init__(self, feature_dim):
        super(Attention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.Tanh(),
            nn.Linear(feature_dim // 2, 1),
            nn.Softmax(dim=1),
        )

    def forward(self, x):
        weights = self.attention(x)
        return (x * weights).sum(dim=1)

In [ ]:
class AttGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=N_CLASSES):
        super(AttGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x, _ = self.gru(x)
        x = self.attention(x)
        x = self.fc(x)
        return x

In [ ]:
# ===================== TRAINING AND EVALUATION ===================== #
def train_model(model, train_loader, test_loader, feature_method, num_epochs=50, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, test_losses, train_accs, test_accs = [], [], [], []

    for epoch in tqdm(range(num_epochs), desc=f"Training AttGRU with {feature_method}"):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        model.eval()
        test_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        test_loss = test_loss / len(test_loader)
        test_acc = correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)

    return train_losses, test_losses, train_accs, test_accs

In [ ]:
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        outputs = model(X_test_tensor)
        _, y_pred = outputs.max(1)
        y_pred = y_pred.cpu().numpy()
        y_prob = torch.softmax(outputs, dim=1).cpu().numpy()

    metrics = {}
    metrics["ACC"] = accuracy_score(y_test, y_pred)
    metrics["AUC"] = roc_auc_score(y_test, y_prob, multi_class="ovr")
    metrics["PRE"] = precision_score(y_test, y_pred, average="macro")
    metrics["SN"] = recall_score(y_test, y_pred, average="macro")
    metrics["F1"] = f1_score(y_test, y_pred, average="macro")
    metrics["MCC"] = matthews_corrcoef(y_test, y_pred)
    return metrics, y_prob, y_pred

In [ ]:
def plot_roc(y_test, y_prob, feature_count):
    y_test_bin = label_binarize(y_test, classes=range(N_CLASSES))  # 4 classes
    fpr, tpr, roc_auc = {}, {}, {}
    plt.figure(figsize=(8, 8))

    for i in range(N_CLASSES):  # Loop over 4 classes
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], label=f"{CLASSES[i]} (AUC = {roc_auc[i]:.2f})")

    plt.plot([0, 1], [0, 1], "k--")
    plt.title(f"AttGRU with {feature_count} Features - ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_PATH, f"AttGRU_SHAP_{feature_count}_roc_curve.png"), dpi=1000, bbox_inches="tight")
    plt.close()

In [ ]:
print("Starting feature selection and AttGRU model training...")

# Define feature counts to compare
feature_counts = [1000, 800, 600, 400, 200]
comparison_results = []

In [ ]:
for n_features in feature_counts:
    print(f"\n===== Evaluating with {n_features} features using SHAP =====")
    X_train_shap, X_test_shap, shap_indices = apply_shap(X_train, X_test, n_features=n_features)

    # Train and evaluate AttGRU model
    batch_size = 64
    num_epochs = 50
    train_dataset = FeatureDataset(X_train_shap, y_train)
    test_dataset = FeatureDataset(X_test_shap, y_test)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = AttGRU(input_dim=X_train_shap.shape[1]).to(device)
    train_losses, test_losses, train_accs, test_accs = train_model(
        model, train_loader, test_loader, feature_method=f"SHAP_{n_features}", num_epochs=num_epochs
    )

    # Evaluate the model
    metrics, y_prob, y_pred = evaluate_model(model, X_test_shap, y_test)
    plot_roc(y_test, y_prob, n_features)

    # Save results
    comparison_results.append(
        {
            "Feature_Count": n_features,
            "ACC": metrics["ACC"],
            "AUC": metrics["AUC"],
            "PRE": metrics["PRE"],
            "SN": metrics["SN"],
            "F1": metrics["F1"],
            "MCC": metrics["MCC"],
        }
    )
    print(f"AttGRU with {n_features} features - Performance Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

In [ ]:
# Save comparison results to CSV
comparison_df = pd.DataFrame(comparison_results)
comparison_df.to_csv(os.path.join(RESULTS_PATH, "AttGRU_SHAP_feature_count_comparison.csv"), index=False)
print(f"Saved comparison results to {os.path.join(RESULTS_PATH, 'AttGRU_SHAP_feature_count_comparison.csv')}")

In [ ]:
# Plot comparison results
plt.figure(figsize=(12, 8))
metrics_to_plot = ["ACC", "AUC", "F1", "MCC"]
x = np.arange(len(feature_counts))
width = 0.2
for i, metric in enumerate(metrics_to_plot):
    values = [result[metric] for result in comparison_results]
    plt.bar(x + i * width, values, width, label=metric)
plt.xlabel("Feature Count")
plt.ylabel("Score")
plt.title("AttGRU Performance with Different Feature Counts (SHAP)")
plt.xticks(x + width * (len(metrics_to_plot) - 1) / 2, feature_counts)
plt.legend()
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_SHAP_feature_count_comparison.png"), dpi=1000, bbox_inches="tight")
plt.close()